# Ruido Cuántico y Decoherencia

**Objetivo:** modelar los principales canales de ruido cuántico y medir su impacto en la fidelidad.

El ruido es el principal obstáculo en la computación cuántica actual. Los canales cuánticos se representan como operadores de Kraus: ε(ρ) = Σ_k K_k ρ K_k†  con Σ_k K_k†K_k = I.

In [ ]:
import numpy as np

I = np.eye(2, dtype=complex)
X = np.array([[0,1],[1,0]], dtype=complex)
Y = np.array([[0,-1j],[1j,0]], dtype=complex)
Z = np.array([[1,0],[0,-1]], dtype=complex)

def apply_channel(rho, kraus_ops):
    return sum(K @ rho @ K.conj().T for K in kraus_ops)

# Estado inicial |+⟩
psi_plus = np.array([1, 1]) / np.sqrt(2)
rho0 = np.outer(psi_plus, psi_plus.conj())
print('ρ₀ =', rho0.round(3))

## Canal Bit-Flip

Voltea el qubit con probabilidad p: K₀ = √(1-p)·I,  K₁ = √p·X

In [ ]:
def bit_flip_kraus(p):
    return [np.sqrt(1-p)*I, np.sqrt(p)*X]

p_vals = [0, 0.1, 0.25, 0.5]
for p in p_vals:
    rho_out = apply_channel(rho0, bit_flip_kraus(p))
    fid = np.real(np.trace(rho0 @ rho_out))  # approx fidelidad
    print(f'p={p:.2f}: ρ_out={rho_out.round(3)}, F≈{fid:.3f}')

## Canal Depolarizante

El canal depolarizante mezcla el estado con ruido blanco:
ε(ρ) = (1 - 3p/4)ρ + p/4(XρX + YρY + ZρZ)

Es el modelo más común para describir errores de puertas.

In [ ]:
def depolarizing_kraus(p):
    return [
        np.sqrt(1-p)  * I,
        np.sqrt(p/3) * X,
        np.sqrt(p/3) * Y,
        np.sqrt(p/3) * Z,
    ]

print('Canal depolarizante:')
for p in [0, 0.1, 0.5, 1.0]:
    rho_out = apply_channel(rho0, depolarizing_kraus(p))
    # Fidelidad exacta
    sqrt_rho0 = np.array([[np.sqrt(rho0[0,0]), 0],[0, np.sqrt(rho0[1,1])]]) 
    F = np.real(np.trace(rho0 @ rho_out))
    print(f'  p={p:.1f}: F={F:.4f}, Traza={np.trace(rho_out).real:.4f}')

## Decoherencia T1 y T2

En hardware real, los qubits decaen en tiempo T1 (relajación) y pierden coherencia en tiempo T2 (desfase).
- T1: tiempo de vida del qubit (|1⟩ → |0⟩)
- T2 ≤ 2T1: tiempo de coherencia (destruye superposición)

In [ ]:
def amplitude_damping_kraus(gamma):
    """gamma = 1 - exp(-t/T1)"""
    K0 = np.array([[1, 0],[0, np.sqrt(1-gamma)]])
    K1 = np.array([[0, np.sqrt(gamma)],[0, 0]])
    return [K0, K1]

# Evolución del estado |1⟩ bajo T1 decay
one = np.array([[0,0],[0,1]], dtype=complex)  # |1⟩⟨1|
t_vals = np.linspace(0, 5, 50)
T1 = 2.0  # μs

pop1_list = []
for t in t_vals:
    gamma = 1 - np.exp(-t/T1)
    rho_t = apply_channel(one, amplitude_damping_kraus(gamma))
    pop1_list.append(rho_t[1,1].real)  # población en |1⟩

import matplotlib.pyplot as plt
plt.figure(figsize=(7,3))
plt.plot(t_vals, pop1_list, 'steelblue')
plt.plot(t_vals, np.exp(-t_vals/T1), 'r--', label='e^(-t/T1)')
plt.xlabel('Tiempo (μs)'); plt.ylabel('P(|1⟩)')
plt.title('Decaimiento T1'); plt.legend()
plt.tight_layout(); plt.savefig('t1_decay.png', dpi=80); plt.show()

In [ ]:
# TVD y distancia de traza
def trace_distance(rho1, rho2):
    diff = rho1 - rho2
    eigvals = np.linalg.eigvalsh(diff @ diff.conj().T)
    return 0.5 * np.sqrt(np.sum(np.abs(eigvals)))

print('Distancia de traza vs ruido depolarizante:')
for p in [0.0, 0.05, 0.1, 0.2]:
    rho_n = apply_channel(rho0, depolarizing_kraus(p))
    td = trace_distance(rho0, rho_n)
    print(f'  p={p:.2f}: TD = {td:.4f}')

## Ejercicio

1. ¿Qué p depolarizante hace TD = 0.5?
2. Implementa el canal phase-flip: K₀=√(1-p)·I, K₁=√p·Z.
3. ¿Cómo difiere el canal phase-flip del bit-flip en términos de qué información destruye?

In [ ]:
# Tu solución
def phase_flip_kraus(p):
    return [np.sqrt(1-p)*I, np.sqrt(p)*Z]

for p in [0.1, 0.5]:
    rho_pf = apply_channel(rho0, phase_flip_kraus(p))
    print(f'Phase-flip p={p}: ρ_out={rho_pf.round(3)}')

## Próximos pasos

- **Lab:** `Cuadernos/laboratorios/14_densitymatrix_ruido_y_tomografia_guiada.ipynb`
- **Módulo:** `Tutorial/06_ruido_y_hardware/README.md`
- **Siguiente guía:** `11_correccion_de_errores.ipynb`